# Caracterização de Grafos de Escala Livre com Dados do SNAP

## Caso: Wikipedia Talk network

### 1. O Dataset
Iniciamos o estudo de caso obtendo o dataset no repositório SNAP (Stanford Large Network Dataset Collection): https://snap.stanford.edu/data/wiki-Talk.html

*"Nodes in the network represent Wikipedia users and a directed edge from node i to node j represents that user i at least once edited a talk page of user j."*

Com base nesta informação, inferimos que se trata de um **grafo direcional**.

### 2. Geração do Grafo
Utilizamos um script em Java para processar o ficheiro bruto (`WikiTalk.txt`) devido ao seu tamanho massivo (mais de 2.3 milhões de vértices e 5 milhões de arestas). O script utiliza a biblioteca `algs4` para iterar pelo grafo e exportar as métricas locais de grau (entrada e saída) para o ficheiro `metricas_vertices.csv`.

**Código de Processamento (`WikiAnalyzer.java`):**


In [1]:
import edu.princeton.cs.algs4.Digraph;
import java.io.*;

public class WikiAnalyzer {
    public static void main(String[] args) throws IOException {
        int numVertices = 2394385; // Valor do cabeçalho
        Digraph G = new Digraph(numVertices + 1); // +1 por segurança de índice

        BufferedReader br = new BufferedReader(new FileReader("data/WikiTalk.txt"));
        String line;

        while ((line = br.readLine()) != null) {
            if (line.startsWith("#")) continue; // Pula comentários

            String[] nodes = line.split("\\s+");
            int v = Integer.parseInt(nodes[0]);
            int w = Integer.parseInt(nodes[1]);
            G.addEdge(v, w);
        }

        System.out.println("Grafo carregado com sucesso!");
        System.out.println("Número de vértices: " + G.V());
        System.out.println("Número de arestas: " + G.E());

        double V = G.V();
        double E = G.E();
        double densidade = E / (V * (V - 1));
        System.out.println("Densidade do grafo: " + densidade);

        System.out.println("Iniciando exportação de métricas...");

        try (PrintWriter writer = new PrintWriter(new File("results/metricas_vertices.csv"))) {
            writer.println("Vertice;GrauEntrada;GrauSaida");

            for (int v = 0; v < G.V(); v++) {
                int in = G.indegree(v);
                int out = G.outdegree(v);

                if (in > 0 || out > 0) {
                    writer.println(v + ";" + in + ";" + out);
                }

                if (v % 500000 == 0) System.out.println("Processados: " + v);
            }
        } catch (FileNotFoundException e) {
            System.err.println("Erro ao criar o arquivo CSV.");
        }
        System.out.println("Exportação concluída! Verifique o arquivo metricas_vertices.csv");
    }
}

SyntaxError: invalid syntax (1302500836.py, line 2)

### 3. Fórmulas Teóricas e Métricas Básicas
Com base nos dados extraídos, aplicamos as fórmulas fundamentais de Teoria dos Grafos para analisar macroscopicamente a rede.

**1. Densidade da Rede ($D$):**
$$D = \frac{E}{V(V-1)}$$

D =~ 8.758

**2. Grau Médio ($\langle k \rangle$):**
Por ser um grafo direcional, o grau médio de entrada é igual ao grau médio de saída:
$$\langle k_{in} \rangle = \langle k_{out} \rangle = \frac{E}{V} \quad \text{e} \quad \langle k \rangle_{total} = \frac{2E}{V}$$
Calculamos um grau médio de aproximadamente **2,09** ($E \approx 5M, V \approx 2.4M$). Isto significa que, em média, cada utilizador edita o mural de ~2 pessoas e tem o seu mural editado por ~2 pessoas. No entanto, é importante sublinhar que em distribuições de cauda longa, a média aritmética não representa adequadamente a grande massa populacional da rede.



### Cálculo do Clustering Médio por Amostragem
Devido a natureza de cauda longa e ao tamanho da rede, o cálculo exato do coeficiente tem um custo proibitivo.
Por esse motivo, utilizamos a biblioteca `networkx` em Python para realizar uma amostragem aleatória de 10.000 nós.

In [2]:
import pandas as pd
import networkx as nx
import random

df = pd.read_csv('C:/Users/Robs/Documents/Eng. Comp/Grafos/grafos-main/data/WikiTalk.txt', sep='\t', comment='#', names=['FromNodeId', 'ToNodeId'])
G = nx.from_pandas_edgelist(df, 'FromNodeId', 'ToNodeId', create_using=nx.DiGraph())
print(f"Vértices: {G.number_of_nodes()} | Arestas: {G.number_of_edges()}")

tamanho_amostra = 10000
print(f"Sorteando {tamanho_amostra} vértices para amostragem...")
todos_os_nos = list(G.nodes())
nos_amostra = random.sample(todos_os_nos, tamanho_amostra)
clustering_medio = nx.average_clustering(G, nodes=nos_amostra)

print(f"Clustering Médio (Amostra): {clustering_medio:.6f}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/Robs/Documents/Eng. Comp/Grafos/grafos-main/data/WikiTalk.txt'

### 4. Visualização da Distribuição de Graus (Histograma)
Para observar o comportamento da cauda longa (*Heavy Tail*), filtramos os vértices com grau inferior a 40 ($k < 40$). O gráfico evidencia a concentração massiva de utilizadores que realizaram apenas a interação mínima.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('C:/Users/Robs/Documents/Eng. Comp/Grafos/grafos-main/results/metricas_vertices.csv', sep=';')
limite_k = 40

fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# Histogramas para In e Out
for i, tipo in enumerate(['GrauEntrada', 'GrauSaida']):
    dados_focados = df[(df[tipo] > 0) & (df[tipo] < limite_k)][tipo]

    ax[i].hist(dados_focados, bins=limite_k, color='royalblue', edgecolor='black', alpha=0.7)
    ax[i].axvline(dados_focados.mean(), color='red', linestyle='dashed', label=f'Média: {dados_focados.mean():.2f}')

    ax[i].set_title(f'Distribuição de {tipo} (k < {limite_k})')
    ax[i].set_xlabel('Grau (k)')
    ax[i].set_ylabel('Frequência')
    ax[i].legend()

plt.tight_layout()
plt.show()

### 5. Análise Gráfica da Escala Livre (Log-Log)
Para comprovar empiricamente o comportamento Escala Livre, plotamos a função de densidade de probabilidade $P(k)$ em escala log-log. A manifestação de uma reta descendente é o reflexo de uma Lei de Potência ($P(k) \sim k^{-\gamma}$).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import powerlaw

df = pd.read_csv('C:/Users/Robs/Documents/Eng. Comp/Grafos/grafos-main/results/metricas_vertices.csv', sep=';')

fig, ax = plt.subplots(1, 2, figsize=(15, 6))

for i, tipo in enumerate(['GrauEntrada', 'GrauSaida']):
    dados = df[df[tipo] > 0][tipo].values

    # Ajuste para extrair Gamma
    fit = powerlaw.Fit(dados, discrete=True, xmin=1, verbose=False)
    gamma = fit.power_law.alpha

    # Cálculo manual da reta teórica
    bins, counts = np.unique(dados, return_counts=True)
    pk = counts / counts.sum()
    C = pk[0] / (bins[0]**-gamma)
    reta_teorica = C * (bins**-gamma)

    # Plotagem
    ax[i].scatter(bins, pk, color='darkorange', alpha=0.6, s=10, label='Dados Reais')
    ax[i].plot(bins, reta_teorica, color='red', linestyle='--', label=f'Ajuste ($\gamma$ = {gamma:.2f})')

    ax[i].set_xscale('log')
    ax[i].set_yscale('log')
    ax[i].set_title(f'Log-Log: {tipo}')
    ax[i].set_xlabel('k (Log)')
    ax[i].set_ylabel('P(k) (Log)')
    ax[i].legend()

plt.tight_layout()
plt.show()

### 6. Validação Estatística do Modelo (Powerlaw)
A análise estritamente visual é vulnerável a enviesamentos. Utilizamos a biblioteca `powerlaw` para confirmar os parâmetros ($\gamma$, $X_{min}$, teste de Kolmogorov-Smirnov(KS)) e efetuar o rácio de verosimilhança (*p-value*) contra distribuições Lognormais.

In [ ]:
import pandas as pd
import powerlaw

df = pd.read_csv('C:/Users/Robs/Documents/Eng. Comp/Grafos/grafos-main/results/metricas_vertices.csv', sep=';')

for tipo in ['GrauEntrada', 'GrauSaida']:
    dados = df[df[tipo] > 0][tipo].values
    fit = powerlaw.Fit(dados, discrete=True, xmin=1, verbose=False)

    print(f"\n--- ANÁLISE: {tipo} ---")
    print(f"Gamma (Alpha): {fit.power_law.alpha:.4f}")
    print(f"Xmin:          {fit.power_law.xmin}")
    print(f"KS (D):        {fit.power_law.D:.4f}")
    print(f"N na Cauda:    {len(fit.power_law.data)}")

    R, p = fit.distribution_compare('power_law', 'lognormal')
    print(f"p-value (vs Lognormal): {p:.4f}")

### 7. Conclusões do Domínio e a Assimetria da Rede

Os testes estatísticos e os gráficos gerados confirmam na prática a nossa hipótese inicial: a rede de interações da Wikipedia (Talk) é fortemente caracterizada por uma topologia de cauda longa (*Heavy Tail*). No entanto, o teste de verossimilhança, revelou uma diferença fascinante entre quem "fala" e quem "ouve" na plataforma.

**A Matemática da Popularidade (Grau de Entrada):**
Para as conexões recebidas, tivemos uma vitória esmagadora do modelo de **Lei de Potência** (Power Law, com $R = 4702.17$ e $p < 0.01$). Com um $\gamma \approx 2.51$, isso ilustra perfeitamente o modelo de **Ligação Preferencial**. Na vida real, a popularidade não tem limites: a página de um super-administrador pode receber milhares de edições de pessoas diferentes de forma passiva, seguindo perfeitamente a regra geométrica de que "o rico fica mais rico".

**O Limite do Esforço Humano (Grau de Saída):**
Já para as conexões feitas (usuários ativamente editando a página de terceiros), o teste estatístico deu a vitória para a distribuição **Lognormal** ($R = -100.99$ e $p < 0.01$). Logo é possível inferir que o esforço tem limite! Enquanto a popularidade de alguém pode crescer infinitamente, um usuário ativo (ou até um bot de manutenção) possui restrições físicas, temporais e sistêmicas (limite de cliques, necessidade de dormir). A Lognormal captura matematicamente essa "frenagem" (*cut-off*) na extremidade direita do gráfico.

**Resumo Final:**
A rede é "sustentada" por um grupo muito restrito de super-*hubs*, enquanto a gigantesca massa de usuários interage o mínimo possível (como evidenciado no histograma). Comprovamos empiricamente que, embora a rede possua uma estrutura pura de **escala livre** na sua "fama" (entrada), ela esbarra de frente nas limitações físicas do mundo real quando avaliamos a capacidade ativa de interação (saída) dos seus participantes.